# 6.2 Vector Database

A vector database is a specialized system for storing and searching high-dimensional data, such as embeddings from text, images, or audio. In Retrieval-Augmented Generation (RAG), vector databases enable efficient similarity search to retrieve relevant information based on semantic meaning, improving the quality and relevance of generated responses.

In [ ]:
%pip install fastapi uvicorn sentence-transformers faiss-cpu

In [2]:
import joblib
import glob
import faiss
import pandas as pd
import numpy as np

from fastapi import FastAPI
from uvicorn import Config, Server

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

from sentence_transformers import SentenceTransformer

## **TASK: Create a Simple Vector Database using TF-IDF**

In this task, you will build a basic vector database for text data using TF-IDF embeddings and nearest neighbor search.

Steps:
1. Prepare a small collection of text documents.
2. Convert the documents into TF-IDF vectors using scikit-learn.
3. Build a nearest neighbor index to enable similarity search over the vectors.
4. Implement a search function to retrieve the most similar documents for a given query.
5. Save the vectorizer, index, and data for later use.

This workflow demonstrates how vector databases can store and search text embeddings, enabling semantic retrieval for applications like RAG.

In [3]:
# 1. Prepare a small collection of text documents.
data = [
    "Machine learning is a field of artificial intelligence",
    "Neural networks are inspired by the brain",
    "FastAPI is a modern web framework for building APIs",
    "Vector databases enable efficient similarity search",
    "Text embeddings convert documents into numerical vectors",
    "Supervised learning uses labeled data",
    "Unsupervised learning finds patterns without labels",
    "Reinforcement learning trains agents through rewards",
    "Python is a popular language for data science",
    "APIs allow applications to communicate with each other"
]

df = pd.DataFrame({'text': data})
df

,text
0,Machine learning is a field of artificial inte...
1,Neural networks are inspired by the brain
2,FastAPI is a modern web framework for building...
3,Vector databases enable efficient similarity s...
4,Text embeddings convert documents into numeric...
5,Supervised learning uses labeled data
6,Unsupervised learning finds patterns without l...
7,Reinforcement learning trains agents through r...
8,Python is a popular language for data science
9,APIs allow applications to communicate with ea...


#### TF-IDF (Term Frequency – Inverse Document Frequency)

TF-IDF is a numerical statistic that measures how important a word is in a document relative to a collection of documents.
TF-IDF gives high weight to terms that are frequent in one document but rare across the corpus.

- Term Frequency (TF): how often a word appears in a document
- Inverse Document Frequency (IDF): how rare the word is across all documents

Term Frequency:
$\text{tf}(t, d) = \frac{f_{t,d}}{\sum_{t' \in d} f_{t',d}}$

Inverse Document Frequency:
$\text{idf}(t, D) = \log\left(\frac{N}{n_t}\right)$

TF-IDF
$\text{tfidf}(t, d, D) = \text{tf}(t, d) \times \text{idf}(t, D)$

Where
- $f_{t,d}$ = count of term $t$ in document $d$
- $N$ = total number of documents
- $n_t$ = number of documents containing term $t$

#### Example

Corpus of 3 documents:

| Doc ID | Text |
|--------|------|
| D1 | “cat sat on mat” |
| D2 | “dog sat on log” |
| D3 | “cat and dog are friends” |

Consider the term **“cat”**:

- Appears in **D1** and **D3** → $n_{\text{cat}} = 2$  
- Total documents $N = 3$
- $\text{idf}(\text{cat}) = \log(3/2) \approx 0.176$

TF values:

- D1: $1/4 = 0.25$
- D2: $0/4 = 0.00$
- D3: $1/5 = 0.20$

TF-IDF table:

| Document | tf("cat") | idf("cat") | tf-idf("cat") |
|---------|-----------|------------|----------------|
| D1 | 0.25 | 0.176 | ≈ 0.044 |
| D2 | 0.00 | 0.176 | 0.000 |
| D3 | 0.20 | 0.176 | ≈ 0.035 |

In [4]:
# 2. Convert the documents into TF-IDF vectors using scikit-learn.
vectorizer = TfidfVectorizer()
embeddings = vectorizer.fit_transform(df['text'])
embeddings.shape

(10, 59)

In [5]:
# 3. Build a nearest neighbor index to enable similarity search over the vectors.
nn = NearestNeighbors(n_neighbors=3, metric='cosine')
nn.fit(embeddings)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",3
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
Name,Type,Value
effective_metric_ effective_metric_: strMetric used to compute distances to neighbors.,str,'cosine'
effective_metric_params_ effective_metric_params_: dictParameters for the metric used to compute distances to neighbors.,dict,{}


In [6]:
# 4. Implement a search function to retrieve the most similar documents for a given query.
def search(query, k=3):
    
    # Vectorize the query
    vec = vectorizer.transform([query])
    
    # Calculate distances between query and data vectors
    distances, indices = nn.kneighbors(vec, n_neighbors=k)
    
    # Return nearest text chunks and their distance w.r.t. the query
    return pd.DataFrame({
        'result': df.iloc[indices[0]]['text'].values,
        'distance': distances[0]
    })

In [7]:
# Search the vector database
search("How do I build an API?")

,result,distance
0,Machine learning is a field of artificial inte...,1.0
1,Neural networks are inspired by the brain,1.0
2,FastAPI is a modern web framework for building...,1.0


In [8]:
# 5. Save the vectorizer, index, and data for later use.
joblib.dump(vectorizer, "./assets/vectorizer.pkl")
joblib.dump(nn, "./assets/nn.pkl")
df.to_csv("./assets/texts.csv", index=False)

## **TASK: Deploy Vector Database**

In this task, you will deploy your TF-IDF-based vector database as a web API using FastAPI.

Steps:
1. Load the saved TF-IDF vectorizer, nearest neighbor index, and text data.
2. Set up a FastAPI application to serve search requests.
3. Implement an endpoint that accepts a query and returns the most similar documents from the database.
4. Run the API server with Uvicorn to make your vector search accessible over HTTP.

This workflow demonstrates how to turn a local vector database into a deployable service, enabling integration with other applications for semantic search and retrieval.

In [9]:
# 1. Load the saved TF-IDF vectorizer, nearest neighbor index, and text data.
vectorizer = joblib.load("./assets/vectorizer.pkl")
nn = joblib.load("./assets/nn.pkl")
df = pd.read_csv("./assets/texts.csv")

In [10]:
# 2. Set up a FastAPI application to serve search requests.
app = FastAPI()

In [11]:

# 3. Implement an endpoint that accepts a query and returns the most similar documents from the database.
@app.get("/search")
def search(q: str, k: int = 5):
    vec = vectorizer.transform([q])
    distances, indices = nn.kneighbors(vec, n_neighbors=k)

    results = []
    for dist, idx in zip(distances[0], indices[0]):
        results.append({
            "text": df.iloc[idx]['text'],
            "distance": float(dist)
        })
    return {"query": q, "results": results}

In [12]:
# 4. Run the API server with Uvicorn to make your vector search accessible over HTTP.
config = Config(app=app, host="localhost", port=8080, log_level="info")
server = Server(config)

await server.serve()

INFO:     Started server process [32716]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://localhost:8080 (Press CTRL+C to quit)
INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [32716]


## **TASK: Deploy a Semantic Search API with FAISS and Sentence Transformers**

In this task, you will deploy a semantic search API using FAISS and Sentence Transformers to enable fast and meaningful retrieval over large text collections.

Steps:
1. Load and chunk a collection of text documents (e.g., essays).
2. Generate dense vector embeddings for each chunk using a pre-trained Sentence Transformer model.
3. Build a FAISS index for efficient similarity search over the embeddings.
4. Implement a retrieval function to find the most relevant text chunks for a given query.
5. Optionally, deploy the retrieval system as a web API for integration with other applications.

This workflow demonstrates how to build and deploy a scalable semantic search service using modern embedding models and vector databases.

In [13]:
# 1. Load and chunk a collection of text documents (e.g., essays)
files = glob.glob("./data/PaulGrahamEssays/*.txt")
texts = []

for f in files:
    with open(f, "r", encoding="utf-8") as fp:
        texts.append(fp.read())

print(f"Loaded {len(texts)} essays.")

Loaded 3 essays.


In [14]:
# 2. Chunk each document into smaller pieces for embedding
def chunk_text(text, chunk_size=400, overlap=80):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

In [15]:
# 3 Create a DataFrame of all text chunks and their source essay IDs
all_chunks = []
essay_ids = []

for idx, t in enumerate(texts):
    chunks = chunk_text(t, chunk_size=100, overlap=80)
    all_chunks.extend(chunks)
    essay_ids.extend([idx] * len(chunks))

df = pd.DataFrame({
    "essay_id": essay_ids,
    "chunk": all_chunks
})

df

,essay_id,chunk
0,0,January 2017Because biographies of famous scie...
1,0,biographies of famous scientists tend to \nedi...
2,0,s scientists tend to \nedit out their mistakes...
3,0,"\nedit out their mistakes, we underestimate t..."
4,0,"takes, we underestimate the \ndegree of risk t..."
...,...,...
4628,2,"ell, Jessica Livingston, Robert Morris, and Ha..."
4629,2,"ton, Robert Morris, and Harj\nTaggar for readi..."
4630,2,and Harj\nTaggar for reading drafts of this.
4631,2,reading drafts of this.


In [ ]:
# 4. Generate dense vector embeddings for each chunk using Sentence Transformers
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(df['chunk'].tolist(), show_progress_bar=True)
embeddings.shape

In [17]:
# 5. Build a FAISS index for efficient similarity search over the embeddings
dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)

index.add(np.array(embeddings).astype("float32"))
print("FAISS index contains:", index.ntotal, "vectors")

FAISS index contains: 4633 vectors


In [18]:
# 6. Implement a retrieval function to find the most relevant text chunks for a given query
def retrieve(query, k=5):
    q_emb = model.encode([query]).astype("float32")
    distances, indices = index.search(q_emb, k)

    results = []
    for score, idx in zip(distances[0], indices[0]):
        results.append({
            "score": float(score),
            "chunk": df.iloc[idx]["chunk"],
            "essay_id": int(df.iloc[idx]["essay_id"])
        })
    return results

In [19]:
# 7. Test the retrieval function with a sample query
results = retrieve("Why do startups innovate faster?", k=3)
for r in results:
    print("---")
    print("Score:", r["score"])
    print(r["chunk"])

---
Score: 0.6477996110916138
in a large number of
startups is that you get a lot of data about how they work.  From
what we've se
---
Score: 0.6497727632522583

startups is that you get a lot of data about how they work.  From
what we've seen, being good seems
---
Score: 0.689254105091095
 trying to learn the most you could
about startups in the shortest possible time, you couldn't have

